In [1]:
import pandas as pd
import numpy as np
import re, os, gc
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import chromadb

# ── Config ────────────────────────────────────────────────────────────
FILTERED_DATA_PATH = '../data/processed/filtered_complaints.csv'
VECTOR_STORE_DIR   = '../vector_store/chroma'
SAMPLE_SIZE        = 500
CHUNK_SIZE         = 500
BATCH_SIZE         = 32
EMBED_MODEL        = 'sentence-transformers/all-MiniLM-L6-v2'
os.makedirs(VECTOR_STORE_DIR, exist_ok=True)

# ── Load data ─────────────────────────────────────────────────────────
print('Loading data...')
df = pd.read_csv(FILTERED_DATA_PATH)
print(f'Loaded {len(df):,} records')

# ── Stratified sample ─────────────────────────────────────────────────
def stratified_sample(df, col, n, seed=42):
    props = df[col].value_counts(normalize=True)
    parts = []
    for cat, prop in props.items():
        n_cat = max(1, round(prop * n))
        part  = df[df[col] == cat]
        parts.append(part.sample(min(n_cat, len(part)), random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

df_sample = stratified_sample(df, 'product_category', SAMPLE_SIZE)
df_sample = df_sample[['Complaint ID','product_category','Issue',
                        'Company','State','Date received',
                        'Consumer complaint narrative']].copy()
del df; gc.collect()
print(f'Sample: {len(df_sample):,}')
print(df_sample['product_category'].value_counts())

# ── Clean & chunk ─────────────────────────────────────────────────────
def clean(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'xxxx|\bxx\b', '', text.lower())
    text = re.sub(r'[^a-z0-9\s.,!?]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def chunk(text, size=500):
    return [text[i:i+size] for i in range(0, len(text), size) if len(text[i:i+size]) > 50]

print('Chunking...')
all_chunks = []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    pieces = chunk(clean(row['Consumer complaint narrative']))
    for idx, piece in enumerate(pieces):
        all_chunks.append({
            'text':             piece,
            'complaint_id':     str(row.get('Complaint ID', '')),
            'product_category': str(row.get('product_category', '')),
            'issue':            str(row.get('Issue', '')),
            'company':          str(row.get('Company', '')),
            'state':            str(row.get('State', '')),
            'date_received':    str(row.get('Date received', '')),
            'chunk_index':      idx,
        })
print(f'Total chunks: {len(all_chunks):,}')

# ── Embed ─────────────────────────────────────────────────────────────
print('\nLoading embedding model...')
model = SentenceTransformer(EMBED_MODEL)
texts = [c['text'] for c in all_chunks]
print(f'Embedding {len(texts):,} chunks...')
embeddings = model.encode(texts, batch_size=BATCH_SIZE,
                          show_progress_bar=True, normalize_embeddings=True,
                          convert_to_numpy=True)
print(f'Embeddings shape: {embeddings.shape}')

# ── Build ChromaDB ────────────────────────────────────────────────────
print('\nBuilding ChromaDB...')
client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
try:
    client.delete_collection('complaints')
except: pass
collection = client.create_collection('complaints', metadata={'hnsw:space': 'cosine'})

for i in tqdm(range(0, len(all_chunks), 200), desc='Indexing'):
    batch = all_chunks[i:i+200]
    collection.add(
        ids        = [f'chunk_{i+j}' for j in range(len(batch))],
        documents  = [c['text'] for c in batch],
        embeddings = embeddings[i:i+200].tolist(),
        metadatas  = [{k:v for k,v in c.items() if k != 'text'} for c in batch],
    )

print(f'\n✅ Done! {collection.count():,} vectors indexed at {VECTOR_STORE_DIR}')

# ── Quick test ────────────────────────────────────────────────────────
q = 'Why are customers unhappy with credit cards?'
qvec = model.encode([q], normalize_embeddings=True).tolist()
res = collection.query(query_embeddings=qvec, n_results=3,
                       include=['documents','metadatas','distances'])
print(f'\nTest query: "{q}"')
for i, (doc, meta, dist) in enumerate(zip(
    res['documents'][0], res['metadatas'][0], res['distances'][0]), 1):
    print(f'\n[{i}] Score: {1-dist:.3f} | {meta["product_category"]} | {meta["issue"]}')
    print(doc[:200])

Loading data...
Loaded 350,134 records
Sample: 499
product_category
Credit Card        270
Savings Account    200
Personal Loan       27
Money Transfer       2
Name: count, dtype: int64
Chunking...


100%|███████████████████████████████████████████████████████████████████████| 499/499 [00:00<00:00, 1618.68it/s]


Total chunks: 1,448

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding 1,448 chunks...


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

Embeddings shape: (1448, 384)

Building ChromaDB...


Indexing: 100%|███████████████████████████████████████████████████████████████████| 8/8 [00:03<00:00,  2.50it/s]


✅ Done! 1,448 vectors indexed at ../vector_store/chroma

Test query: "Why are customers unhappy with credit cards?"

[1] Score: 0.589 | Credit Card | Billing disputes
my statement arrived and upon review i had noticed charges that i did not make. futhermore, i only made purchase. second, i contacted the company and they stated they would dispute the item. i made on

[2] Score: 0.586 | Savings Account | Managing an account
hat cause a trickle down of issues and stress and for its consumers where nothing is done and loyal people like myself who have been with the company and also have relations with other banking institu

[3] Score: 0.567 | Credit Card | Closing your account
initial complaint was due to amex reducing credit limit over with consumer never missing a payment to them. instead of reclaiming the credit limit with interest. amex sent me a new credit card without
